# Elicitation Robustness Across Scale

Measures mean token entropy for accessibility concepts and a control concept across 
the full Pythia model suite and GPT-2 model suite, using five distinct elicitation 
strategies. If the declarative-evaluative gap is real and not a prompting artifact, 
all template types should show the same emergence threshold.

**Pythia models:** 160M, 410M, 1B, 2.8B, 6.9B  
**GPT-2 models:** small (124M), medium (355M), large (774M), xl (1.5B)  
**Concepts:** closed captions, color contrast, page title, bicycle (control)  
**Template types:** cloze, direct_question, instruction, evaluative, scenario  
**Metric:** Mean token entropy (bits) — lower = more confident

## Setup

**Run the install cell, then restart the runtime (Runtime → Restart session), then continue from the imports cell.** The numpy pin is needed to fix a Colab 3.12 compatibility issue with transformers.

In [ ]:
!pip install numpy==1.26.4 transformer_lens -q

⚠️ **Restart runtime now** (Runtime → Restart session), then run from the next cell.

In [ ]:
import json
import torch
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from transformer_lens import HookedTransformer
from google.colab import files
import gc

print(f'numpy: {np.__version__}')
print(f'torch: {torch.__version__}')

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## Load Prompts

In [ ]:
# Upload elicitation_control_prompts.jsonl
uploaded = files.upload()

In [ ]:
# Parse prompts
all_prompts = []
with open('elicitation_control_prompts.jsonl') as f:
    for line in f:
        all_prompts.append(json.loads(line.strip()))

template_types = ['cloze', 'direct_question', 'instruction', 'evaluative', 'scenario']
concepts = ['closed captions', 'color contrast', 'page title', 'bicycle']

MAX_TOKENS = 50

print(f'Loaded {len(all_prompts)} prompts')
print(f'Template types: {template_types}')
print(f'Concepts: {concepts}')

## Entropy Computation

In [ ]:
def get_token_entropy(model, text):
    """Mean per-token entropy of the model's predictions."""
    tokens = model.to_tokens(text)
    logits = model(tokens)
    probs = torch.nn.functional.softmax(logits[0, :-1, :], dim=-1)
    entropy = -(probs * torch.log2(probs + 1e-10)).sum(dim=-1)
    return entropy.mean().item()


def run_model(model_name, display_name, all_prompts, max_tokens):
    """Load model, run all prompts, compute entropy, return results, cleanup."""
    print(f'\n{"=" * 60}')
    print(f'Loading {display_name}...')
    print(f'{"=" * 60}')
    
    model = HookedTransformer.from_pretrained(model_name, device=device)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f'Params: {n_params:.1f}M')
    
    results = []
    for i, p in enumerate(all_prompts):
        output = model.generate(p['prompt'], max_new_tokens=max_tokens, temperature=0, verbose=False)
        completion = output[len(p['prompt']):]
        entropy = get_token_entropy(model, output)
        
        results.append({
            'model': model_name,
            'display_name': display_name,
            'n_params': n_params,
            'concept': p['concept'],
            'template_type': p['template_type'],
            'prompt_id': p['prompt_id'],
            'prompt': p['prompt'],
            'completion': completion,
            'word_count': len(completion.split()),
            'entropy': entropy
        })
        print(f'  [{i+1:2d}/{len(all_prompts)}] {p["concept"]:16} | {p["template_type"]:16} | entropy: {entropy:.2f}')
    
    del model
    gc.collect()
    torch.cuda.empty_cache()
    print(f'{display_name} done — memory cleared')
    
    return results

## Run Pythia Suite

160M → 410M → 1B → 2.8B → 6.9B

In [ ]:
pythia_models = [
    ('pythia-160m', 'Pythia 160M'),
    ('pythia-410m', 'Pythia 410M'),
    ('pythia-1b', 'Pythia 1B'),
    ('pythia-2.8b', 'Pythia 2.8B'),
    ('pythia-6.9b', 'Pythia 6.9B'),
]

all_results = []
for model_name, display_name in pythia_models:
    results = run_model(model_name, display_name, all_prompts, MAX_TOKENS)
    all_results.extend(results)
    
print(f'\nPythia suite complete: {len(all_results)} results')

## Run GPT-2 Suite

Small (124M) → Medium (355M) → Large (774M) → XL (1.5B)

In [ ]:
gpt2_models = [
    ('gpt2-small', 'GPT-2 Small'),
    ('gpt2-medium', 'GPT-2 Medium'),
    ('gpt2-large', 'GPT-2 Large'),
    ('gpt2-xl', 'GPT-2 XL'),
]

for model_name, display_name in gpt2_models:
    results = run_model(model_name, display_name, all_prompts, MAX_TOKENS)
    all_results.extend(results)
    
print(f'\nAll models complete: {len(all_results)} results')

## Save Results

In [ ]:
df = pd.DataFrame(all_results)
df.to_csv('elicitation_robustness_scaling.csv', index=False)
print(f'Saved {len(df)} results')
print(f'Models: {df["display_name"].unique().tolist()}')
df.head()

## Figure: Entropy Across Scale

The key visualization. If the elicitation robustness claim holds, all five 
template type lines should show the same inflection at 2.8B for accessibility 
concepts, while bicycle stays flat.

In [ ]:
# Compute mean entropy: accessibility concepts averaged, per template type per model
pythia_df = df[df['model'].str.startswith('pythia')]

# Accessibility concepts only
acc_df = pythia_df[pythia_df['concept'] != 'bicycle']
acc_by_template = acc_df.groupby(['n_params', 'template_type'])['entropy'].mean().reset_index()

# Bicycle control averaged across template types
bike_df = pythia_df[pythia_df['concept'] == 'bicycle']
bike_mean = bike_df.groupby('n_params')['entropy'].mean().reset_index()

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

colors = {'cloze': '#1b9e77', 'direct_question': '#d95f02', 'instruction': '#7570b3', 
          'evaluative': '#e7298a', 'scenario': '#66a61e'}

for tt in template_types:
    tt_data = acc_by_template[acc_by_template['template_type'] == tt].sort_values('n_params')
    ax.plot(tt_data['n_params'], tt_data['entropy'], 'o-', 
            label=tt.replace('_', ' '), color=colors[tt], linewidth=2, markersize=6)

# Bicycle control
bike_mean_sorted = bike_mean.sort_values('n_params')
ax.plot(bike_mean_sorted['n_params'], bike_mean_sorted['entropy'], 
        's--', color='gray', linewidth=2, markersize=6, label='bicycle (control)', alpha=0.7)

# 2.8B threshold line
ax.axvline(x=2800, color='gray', linestyle=':', alpha=0.5)
ax.annotate('2.8B emergence\nthreshold', xy=(2800, ax.get_ylim()[1] * 0.95), 
            fontsize=9, ha='center', color='gray')

ax.set_xlabel('Model Size (M parameters)', fontsize=12)
ax.set_ylabel('Mean Token Entropy (bits)', fontsize=12)
ax.set_title('Elicitation Robustness: Entropy Across Pythia Scale\nby Template Type (accessibility concepts averaged)', 
             fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.set_xscale('log')
ax.set_xticks([160, 410, 1000, 2800, 6900])
ax.set_xticklabels(['160M', '410M', '1B', '2.8B', '6.9B'])

plt.tight_layout()
plt.savefig('elicitation_robustness_pythia_entropy.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved elicitation_robustness_pythia_entropy.png')

### Accessibility vs Control Delta

Entropy difference between accessibility concepts and bicycle control, 
per model size. Positive = model is less confident about accessibility 
than about bicycles. If the gap closes at 2.8B, the delta should 
approach zero at the emergence threshold.

In [ ]:
# Compute delta: accessibility entropy minus bicycle entropy, per model per template type
acc_pivot = acc_df.groupby(['n_params', 'template_type'])['entropy'].mean().reset_index()
bike_pivot = bike_df.groupby(['n_params', 'template_type'])['entropy'].mean().reset_index()

delta_rows = []
for _, acc_row in acc_pivot.iterrows():
    bike_row = bike_pivot[(bike_pivot['n_params'] == acc_row['n_params']) & 
                           (bike_pivot['template_type'] == acc_row['template_type'])]
    if len(bike_row) > 0:
        delta_rows.append({
            'n_params': acc_row['n_params'],
            'template_type': acc_row['template_type'],
            'delta_entropy': acc_row['entropy'] - bike_row.iloc[0]['entropy']
        })

delta_df = pd.DataFrame(delta_rows)

fig, ax = plt.subplots(figsize=(10, 6))

for tt in template_types:
    tt_data = delta_df[delta_df['template_type'] == tt].sort_values('n_params')
    ax.plot(tt_data['n_params'], tt_data['delta_entropy'], 'o-',
            label=tt.replace('_', ' '), color=colors[tt], linewidth=2, markersize=6)

ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
ax.axvline(x=2800, color='gray', linestyle=':', alpha=0.5)
ax.annotate('2.8B emergence\nthreshold', xy=(2800, ax.get_ylim()[1] * 0.9),
            fontsize=9, ha='center', color='gray')

ax.set_xlabel('Model Size (M parameters)', fontsize=12)
ax.set_ylabel('Entropy Delta (accessibility − bicycle, bits)', fontsize=12)
ax.set_title('Elicitation Robustness: Accessibility Entropy Premium\nby Template Type Across Pythia Scale',
             fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.set_xscale('log')
ax.set_xticks([160, 410, 1000, 2800, 6900])
ax.set_xticklabels(['160M', '410M', '1B', '2.8B', '6.9B'])

plt.tight_layout()
plt.savefig('elicitation_robustness_delta.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved elicitation_robustness_delta.png')

### GPT-2 Cross-Architecture Comparison

In [ ]:
gpt2_df = df[df['model'].str.startswith('gpt2')]

acc_gpt2 = gpt2_df[gpt2_df['concept'] != 'bicycle']
acc_gpt2_by_tt = acc_gpt2.groupby(['n_params', 'template_type'])['entropy'].mean().reset_index()

bike_gpt2 = gpt2_df[gpt2_df['concept'] == 'bicycle']
bike_gpt2_mean = bike_gpt2.groupby('n_params')['entropy'].mean().reset_index()

fig, ax = plt.subplots(figsize=(10, 6))

for tt in template_types:
    tt_data = acc_gpt2_by_tt[acc_gpt2_by_tt['template_type'] == tt].sort_values('n_params')
    ax.plot(tt_data['n_params'], tt_data['entropy'], 'o-',
            label=tt.replace('_', ' '), color=colors[tt], linewidth=2, markersize=6)

bike_sorted = bike_gpt2_mean.sort_values('n_params')
ax.plot(bike_sorted['n_params'], bike_sorted['entropy'],
        's--', color='gray', linewidth=2, markersize=6, label='bicycle (control)', alpha=0.7)

ax.set_xlabel('Model Size (M parameters)', fontsize=12)
ax.set_ylabel('Mean Token Entropy (bits)', fontsize=12)
ax.set_title('Elicitation Robustness: Entropy Across GPT-2 Scale\nby Template Type (accessibility concepts averaged)',
             fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.set_xscale('log')
ax.set_xticks([124, 355, 774, 1500])
ax.set_xticklabels(['Small\n124M', 'Medium\n355M', 'Large\n774M', 'XL\n1.5B'])

plt.tight_layout()
plt.savefig('elicitation_robustness_gpt2_entropy.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved elicitation_robustness_gpt2_entropy.png')

## Download Results

In [ ]:
files.download('elicitation_robustness_scaling.csv')
files.download('elicitation_robustness_pythia_entropy.png')
files.download('elicitation_robustness_delta.png')
files.download('elicitation_robustness_gpt2_entropy.png')